# Writing New Molecules: the Math and Data Behind a Transformer Autoencoder
**Series B: Deep Learning for Molecules, Post 6** &nbsp;|&nbsp; companion *primer* notebook

This is the lightweight companion to the full tutorial notebook `VAE_Tutorial.ipynb` (which needs a GPU and the 88 MB pre-trained model).
**This primer needs neither.** It runs on a CPU in about a minute and lets you check, with your own hands, the data facts and the mathematics behind the post:

| Section | What you verify |
|---|---|
| 1 | tokenising 249,455 SMILES strings; padding; the train/test split |
| 2 | where all 21,859,969 parameters of the model live |
| 3 | why attention is divided by $\sqrt{d_k}$ |
| 4 | the geometry of a 1280-dimensional Gaussian |
| 5 | what the model's *simplified MMD penalty* really enforces |
| 6 | why random sampling works anyway |
| 7 | validity, uniqueness and novelty with error bars |
| 8 | why straight-line interpolation in latent space shrinks the vectors |

**How to use it:** run top to bottom (**Runtime → Run all**), try the **Stop and predict** cells, then do the exercises.
The chemistry numbers (validity, the ibuprofen to caffeine path, ...) come from the full notebook; here they are typed in from its saved run, and we analyse them.

## 0. Setup and data

In [ ]:
import os, math, urllib.request
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

CSV_NAME = "250k_rndm_zinc_drugs_clean_3.csv"
CSV_URL = ("https://raw.githubusercontent.com/aspuru-guzik-group/chemical_vae/master/"
           "models/zinc_properties/250k_rndm_zinc_drugs_clean_3.csv")

if not os.path.exists(CSV_NAME):
    try:
        print("Downloading the ZINC data set (about 22 MB) ...")
        urllib.request.urlretrieve(CSV_URL, CSV_NAME)
    except Exception as err:
        print("Download failed:", err)
        try:
            from google.colab import files
            print("Please select", CSV_NAME)
            up = files.upload(); os.replace(next(iter(up)), CSV_NAME)
        except ImportError:
            raise FileNotFoundError(f"Place {CSV_NAME} next to this notebook and re-run.")

csv = pd.read_csv(CSV_NAME)
print(csv.shape, list(csv.columns))

## 1. From SMILES strings to tokens
Same recipe as the full tutorial: add a start marker `!` and an end marker `?`, give every distinct character an integer (in order of first appearance), pad to the longest string.
**Stop and predict:** the longest string has 111 tokens and the median has 46. What fraction of the padded matrix is padding?

In [ ]:
raw = [s.strip() for s in csv.smiles.values]            # the file stores a newline after every string
data = [['!'] + list(s) + ['?'] for s in raw]

smiledict, invdict = {}, {}
for seq in data:
    for ch in seq:
        if ch not in smiledict:
            smiledict[ch] = len(smiledict); invdict[smiledict[ch]] = ch
pad_index = len(smiledict)
lengths = np.array([len(s) for s in data]); maxlength = int(lengths.max())

print(f"{len(data):,} molecules | {len(smiledict)} distinct characters | padding token = {pad_index}")
print(f"tokens per molecule: min {lengths.min()}, median {int(np.median(lengths))}, mean {lengths.mean():.1f}, max {maxlength}")
print(f"padding share of the padded matrix: {100 * (1 - lengths.mean() / maxlength):.1f}%")
print(f"strings with at most 60 tokens: {100 * (lengths <= 60).mean():.1f}%   |   with 100 or more: {(lengths >= 100).sum()}")

# split exactly as in the full tutorial
batch_size = 256
testsize = math.ceil(len(data) / 10); n_train = len(data) - testsize
n_batches = len(range(0, n_train - batch_size, batch_size))
print(f"train {n_train:,} | test {testsize:,} | {n_batches} training batches of {batch_size} "
      f"(the last {n_train - n_batches * batch_size} training molecules are dropped)")

first = [smiledict[c] for c in data[0]]
print("\nfirst molecule:", raw[0])
print("its tokens    :", first, f"+ {maxlength - len(first)} x {pad_index} (padding)")

plt.figure(figsize=(7, 3.4))
plt.hist(lengths, bins=np.arange(lengths.min(), lengths.max() + 2) - 0.5, color=COLORS["blue"])
plt.axvline(maxlength, color="k", ls="--", lw=1); plt.xlabel("tokens per molecule"); plt.ylabel("molecules")
plt.title("Most of each padded row is padding"); plt.show()

## 2. Where do the 21,859,969 parameters live?
The full tutorial prints the total. Let us *derive* it from the architecture: 37 tokens, embedding size 128, 6 layers each, 4 heads, feed-forward width 256, a 10 × 128 latent vector.
A linear layer from $m$ to $n$ units has $mn+n$ parameters; a layer-norm has $2n$.

In [ ]:
V, H, PF, LAYERS, BL = 37, 128, 256, 6, 10        # vocabulary, embedding, feed-forward width, layers, latent vectors
POS, MAXLEN = 120, 111                            # positions in the embedding table, padded string length

lin = lambda m, n: m * n + n
ln  = lambda n: 2 * n
attn = 4 * lin(H, H)                              # query, key, value and output projections
ff   = lin(H, PF) + lin(PF, H)
emb  = V * H + POS * H                            # token + position embeddings

parts = {
    "encoder embeddings":            emb,
    "encoder layers (6)":            LAYERS * (2 * ln(H) + attn + ff),
    "decoder embeddings":            emb,
    "decoder layers (6)":            LAYERS * (3 * ln(H) + 2 * attn + ff),   # self- and cross-attention
    "decoder output layer":          lin(H, V),
    "latent layer (14,208 -> 1,280)": lin(MAXLEN * H, BL * H),
    "inverse layer (1,280 -> 1,280)": lin(BL * H, BL * H),
    "2 residual blocks":             2 * lin(BL, BL),
}
total = sum(parts.values())
for k, v in parts.items():
    print(f"{k:34s} {v:>12,d}  {100 * v / total:5.1f}%")
print(f"{'TOTAL':34s} {total:>12,d}")
print("matches the number printed by the full tutorial (21,859,969):", total == 21_859_969)

## 3. Why attention divides by $\sqrt{d_k}$
Attention scores are dot products $q\cdot k$ of vectors with $d_k=32$ entries (128 / 4 heads).
**Stop and predict:** if the entries of $q$ and $k$ have mean 0 and variance 1, what is the variance of $q\cdot k$?

In [ ]:
rng = np.random.default_rng(0)
dk, n_keys = 32, 111
q = rng.standard_normal((20000, dk)); k = rng.standard_normal((20000, dk))
print("variance of q.k            :", round(((q * k).sum(1)).var(), 2), f"(theory: d_k = {dk})")
print("variance of q.k / sqrt(d_k):", round(((q * k).sum(1) / np.sqrt(dk)).var(), 2))

def mean_max_attention(scale, trials=2000):
    Q = rng.standard_normal((trials, dk)); K = rng.standard_normal((n_keys, dk))
    e = Q @ K.T / scale; e -= e.max(1, keepdims=True)
    p = np.exp(e); p /= p.sum(1, keepdims=True)
    return p.max(1).mean()

print(f"\naverage largest attention weight over {n_keys} keys:")
print(f"  without scaling : {mean_max_attention(1.0):.3f}   (one key gets most of the weight)")
print(f"  with sqrt(d_k)  : {mean_max_attention(np.sqrt(dk)):.3f}   (uniform would be {1 / n_keys:.3f})")

## 4. The geometry of a 1280-dimensional Gaussian
A molecule is described by $d=10\times128=1280$ latent numbers. A draw $y\sim N(0,I_d)$ does *not* land near the origin: its length is almost always close to $\sqrt d\approx35.8$.
We also need the **null reference** for the principal-component test of the full tutorial: with $n$ points in $d$ dimensions, the first principal component of perfect Gaussian data has standard deviation about $1+\sqrt{d/n}$, not 1.

In [ ]:
d = 1280
rng = np.random.default_rng(0)
norms = np.linalg.norm(rng.standard_normal((20000, d)), axis=1)
print(f"length of N(0, I_{d}) draws: mean {norms.mean():.2f}, std {norms.std():.2f}   (sqrt(d) = {math.sqrt(d):.2f})")

print("\nfirst principal component of perfect N(0, I) data:")
for n in (256, 2000):
    A = rng.standard_normal((n, d)); A -= A.mean(0)
    sd1 = np.linalg.svd(A, compute_uv=False)[0] / math.sqrt(n - 1)
    print(f"  n = {n:>4}: std {sd1:.2f}   (prediction 1 + sqrt(d/n) = {1 + math.sqrt(d / n):.2f})")

## 5. What does the model's "MMD" penalty really enforce?
The loss is cross-entropy + `mmd(latent, prior_samples)`. The implementation uses the kernel
$$k(x,y)=\exp\!\Big(-\tfrac{1}{d\cdot0.0005}\cdot\tfrac1d\lVert x-y\rVert^2\Big)=\exp\!\big(-\lVert x-y\rVert^2/s\big),\qquad s=d^2\cdot0.0005=819.2,$$
and the *simplified* penalty $8\,[1-\overline{k(z,y)}]$, which keeps only the **cross term** of the MMD (the full estimator also has $E\,k(z,z')$ and $E\,k(y,y')$; that is Exercise 1 of the full tutorial).

For $z\sim N(0,\sigma^2I)$ and $y\sim N(0,I)$ everything has a closed form (Supplement S4 of the post):
$$E\,k(z,y)=\Big(1+\tfrac{2(\sigma^2+1)}{s}\Big)^{-d/2}.$$
**Stop and predict:** as the latent vectors shrink toward the origin ($\sigma\to0$), does the simplified penalty go up or down? And the full MMD²?

In [ ]:
d, c = 1280, 0.0005
s = d * d * c

def kern(X, Y):                                   # the notebook's kernel, via ||x-y||^2 = |x|^2 + |y|^2 - 2 x.y
    D = (X**2).sum(1)[:, None] + (Y**2).sum(1)[None, :] - 2 * X @ Y.T
    return np.exp(-np.maximum(D, 0) / d / (d * c))

cross = lambda sx, sy=1.0: (1 + 2 * (sx**2 + sy**2) / s) ** (-d / 2)
simple_analytic = lambda sx: 8 * (1 - cross(sx))
full_analytic   = lambda sx, sy=1.0: (1 + 4*sx**2/s) ** (-d/2) + (1 + 4*sy**2/s) ** (-d/2) - 2 * cross(sx, sy)

rng = np.random.default_rng(0)
Y = rng.standard_normal((1000, d))                # 1000 prior samples, as in the training loop
print(f"{'sigma':>6} | {'simplified (MC)':>15} {'(analytic)':>11} | {'full MMD^2 (MC)':>15} {'(analytic)':>11}")
for sx in (0.0, 0.2, 0.39, 0.7, 1.0, 1.5):
    Z = sx * rng.standard_normal((1000, d)); n = len(Z)
    kzy = kern(Z, Y).mean()
    kzz = kern(Z, Z); kzz = (kzz.sum() - np.trace(kzz)) / (n * (n - 1))
    kyy = kern(Y, Y); kyy = (kyy.sum() - np.trace(kyy)) / (n * (n - 1))
    print(f"{sx:>6.2f} | {8 * (1 - kzy):>15.3f} {simple_analytic(sx):>11.3f} | {kzz + kyy - 2 * kzy:>+15.5f} {full_analytic(sx):>+11.5f}")

In [ ]:
sig = np.linspace(0, 1.6, 200)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(sig, simple_analytic(sig), color=COLORS["red"], lw=2)
ax[0].axhline(7.641, color=COLORS["gray"], ls="--"); ax[0].text(0.02, 7.66, "prior vs prior (measured 7.641)", fontsize=8, color=COLORS["gray"])
ax[0].axhline(6.672, color=COLORS["blue"], ls="--"); ax[0].text(0.02, 6.69, "encoded molecules vs prior (measured 6.672)", fontsize=8, color=COLORS["blue"])
ax[0].set(xlabel="spread sigma of the latent vectors", ylabel="simplified penalty", title="(a) Simplified (cross-term) penalty")
ax[1].plot(sig, full_analytic(sig), color=COLORS["green"], lw=2); ax[1].axvline(1, color="k", ls=":", lw=1)
ax[1].set(xlabel="spread sigma of the latent vectors", ylabel="full MMD$^2$", title="(b) Full MMD$^2$ (zero only at sigma = 1)")
plt.tight_layout(); plt.show()

# at which sigma does the simplified penalty equal the value the full tutorial measured on real encoded molecules?
lo, hi = 0.0, 1.0
for _ in range(60):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if simple_analytic(mid) < 6.672 else (lo, mid)
print(f"simplified penalty = 6.672  <=>  sigma = {lo:.3f}   (the full tutorial measured a mean coordinate std of 0.39)")

**What you just saw.** The cross-term-only penalty keeps *decreasing* as the latent vectors shrink toward the origin, while the full MMD² has its minimum at $\sigma=1$. So the simplified penalty pulls the encoded vectors **inward** (toward the point $0$), not toward the shape of the prior. The values measured in the full tutorial (penalty 6.672 against a reference of 7.641, mean $\lVert z\rVert=14.3$ against 35.8, mean coordinate std 0.39 against 1) are exactly what this picture predicts.

## 6. Then why does random sampling work?
During training the decoder never sees $z$ itself but $z+\varepsilon$ with $\varepsilon\sim N(0,I)$ (the noise added in `Seq2Seq.forward`). If the encoder's coordinates have spread $\sigma_z\approx0.39$, what does the decoder's *training input* look like, compared with a draw from the prior used for sampling?

In [ ]:
sz = 0.39
rows = [("encoder output z (measured mean |z| = 14.3)",   math.sqrt(d) * sz),
        ("decoder TRAINING input z + noise",               math.sqrt(d * (sz**2 + 1))),
        ("decoder SAMPLING input: a draw from N(0, I)",    math.sqrt(d))]
for name, r in rows:
    print(f"{name:50s} length ~ {r:5.1f}")
print(f"\nper-coordinate std of z + noise: {math.sqrt(sz**2 + 1):.3f}  (prior: 1)")

The noise makes the decoder's training inputs look almost like prior samples (length about 38 against 36), so decoding prior samples is *close* to what the decoder practised. This is a hypothesis that fits the numbers; we have not tested it on the model (see Exercise 5 of the post).

## 7. Validity, uniqueness, novelty, with error bars
The counts below are typed in from the saved run of the full tutorial (5000 random latent vectors; 5000 test molecules).

In [ ]:
def percent_with_ci(k, n):
    p = k / n
    return 100 * p, 100 * 1.96 * math.sqrt(p * (1 - p) / n)          # normal-approximation 95% interval

rows = [("Parses as SMILES (grammar only)",        4347, 5000),
        ("Chemically valid",                        3971, 5000),
        ("Repeated (among valid)",                  161,  3971),
        ("Novel (among distinct valid)",            3803, 3810),
        ("Exactly reconstructed (test molecules)",  3429, 5000)]
table = pd.DataFrame([(m, k, n, *percent_with_ci(k, n)) for m, k, n in rows],
                     columns=["Metric", "count", "out of", "percent", "95% CI (+/- points)"])
print(table.round(2).to_string(index=False))

valid, distinct, novel = 3971, 3971 - 161, 3803
print(f"\nuniqueness = {100 * distinct / valid:.2f}%")
print(f"overall yield of NEW, DISTINCT, VALID molecules = {novel}/5000 = {100 * novel / 5000:.2f}%"
      f"  (= validity x uniqueness x novelty = {100 * (valid/5000) * (distinct/valid) * (novel/distinct):.2f}%)")

fail = {"valid": 79.42, "syntax error": 13.06, "Kekulize": 7.20, "valence": 0.28, "AtomKekulize": 0.04}
plt.figure(figsize=(7, 3.0))
left = 0
for (k_, v_), col in zip(fail.items(), [COLORS["green"], COLORS["red"], COLORS["orange"], COLORS["pink"], COLORS["gray"]]):
    plt.barh([0], [v_], left=left, color=col, label=f"{k_} ({v_:.2f}%)"); left += v_
plt.yticks([]); plt.xlabel("% of 5000 sampled strings"); plt.legend(fontsize=7, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.3))
plt.title("What happens to 5000 random samples"); plt.tight_layout(); plt.show()

## 8. Why does a straight line in latent space leave the "typical" region?
The full tutorial interpolates **linearly** between the latent vectors of ibuprofen and caffeine. Here is what linear interpolation does to the *length* of a vector in high dimension, for two independent prior-like endpoints, compared with **spherical** interpolation (slerp).

In [ ]:
rng = np.random.default_rng(0)
a = rng.standard_normal((5000, d)); b = rng.standard_normal((5000, d))
def slerp(a, b, t):
    w = np.arccos((a * b).sum(1) / np.linalg.norm(a, axis=1) / np.linalg.norm(b, axis=1))[:, None]
    return (np.sin((1 - t) * w) * a + np.sin(t * w) * b) / np.sin(w)

ts = np.linspace(0, 1, 11)
lerp_norm  = [np.linalg.norm((1 - t) * a + t * b, axis=1).mean() for t in ts]
slerp_norm = [np.linalg.norm(slerp(a, b, t), axis=1).mean() for t in ts]
theory = [math.sqrt(d) * math.sqrt((1 - t)**2 + t**2) for t in ts]

print(f"{'t':>4} {'lerp':>7} {'theory':>7} {'slerp':>7}")
for t, l, th, sl in zip(ts, lerp_norm, theory, slerp_norm):
    print(f"{t:>4.1f} {l:>7.2f} {th:>7.2f} {sl:>7.2f}")

plt.figure(figsize=(6.5, 3.6))
plt.plot(ts, lerp_norm, "o-", color=COLORS["red"], label="linear interpolation")
plt.plot(ts, slerp_norm, "s-", color=COLORS["blue"], label="spherical interpolation (slerp)")
plt.axhline(math.sqrt(d), color=COLORS["gray"], ls="--"); plt.ylim(0, 40)
plt.xlabel("t (0 = first endpoint, 1 = second)"); plt.ylabel("length of the vector"); plt.legend(fontsize=8)
plt.title("The midpoint of a straight line is too short"); plt.tight_layout(); plt.show()

## 9. Your turn
**Exercise 1 (attention).** Prove that $\mathrm{Var}(q\cdot k)=d_k$ for independent zero-mean unit-variance entries, then check with the code of Section 3 for $d_k=8$ and $d_k=128$.

**Exercise 2 (parameters).** The *latent layer* holds what fraction of all parameters? What would that fraction be with 5 latent vectors instead of 10?

**Exercise 3 (the penalty).** Use the closed form to compute the simplified penalty for $\sigma=0.5$ and $\sigma=2$. Which is smaller? Is that what a good generative model wants?

**Exercise 4 (error bars).** Compute the 95% interval for the *uniqueness* (95.95%). Then: how many samples would you need to halve the interval for validity?

**Exercise 5 (yield).** At the measured yield of 76.06% new distinct valid molecules per sample, how many draws give 10,000 of them? Why is this an optimistic estimate? (Hint: what happens to repeats as the sample grows?)

**Exercise 6 (interpolation).** Derive the theory column of Section 8, $\sqrt d\sqrt{(1-t)^2+t^2}$. At which $t$ is the shrinkage largest, and by what factor?

In [ ]:
# Starters
# Exercise 2
lat = lin(MAXLEN * H, BL * H)
print("latent share with 10 vectors: %.1f%%" % (100 * lat / total))
alt = total - parts["latent layer (14,208 -> 1,280)"] - parts["inverse layer (1,280 -> 1,280)"] - parts["2 residual blocks"] \
      + lin(MAXLEN * H, 5 * H) + lin(5 * H, 5 * H) + 2 * lin(5, 5)
print("with 5 vectors: total %s, latent share %.1f%%" % (f"{alt:,}", 100 * lin(MAXLEN * H, 5 * H) / alt))

# Exercise 3
print("simplified penalty: sigma=0.5 ->", round(simple_analytic(0.5), 3), "| sigma=2 ->", round(simple_analytic(2.0), 3))

# Exercise 4
print("uniqueness: %.2f +/- %.2f points" % percent_with_ci(3810, 3971))

# Exercise 5
yield_ = 3803 / 5000
print("draws for 10,000 new distinct valid molecules (optimistic):", math.ceil(10_000 / yield_))

## 10. Self-check

In [ ]:
assert len(csv) == 249_455 and len(smiledict) == 36 and maxlength == 111 and int(np.median(lengths)) == 46
assert (n_train, testsize, n_batches) == (224_509, 24_946, 876)
assert first[:16] == [0, 1, 1, 2, 1, 3, 2, 1, 3, 4, 5, 4, 4, 4, 6, 7]       # same tokens as the full tutorial prints
assert total == 21_859_969
assert abs(simple_analytic(1.0) - 7.641) < 0.01 and abs(simple_analytic(0.39) - 6.672) < 0.01
assert simple_analytic(0.0) < simple_analytic(0.39) < simple_analytic(1.0) and abs(full_analytic(1.0)) < 1e-9
assert abs(norms.mean() - 35.77) < 0.05 and 24.5 < lerp_norm[5] < 26.0 and abs(slerp_norm[5] - math.sqrt(d)) < 0.2
assert round(percent_with_ci(3971, 5000)[1], 2) == 1.12 and round(percent_with_ci(3429, 5000)[0], 2) == 68.58
print("All checks passed: you reproduced the numbers in the post.")